# 🌐 MonoSplat — Gaussian Splat Training on Kaggle GPU

## Before you run this notebook

This notebook only handles **Gaussian Splat training**. FFmpeg and COLMAP already ran on your desktop.

You need to have already:
- ✅ Started the MonoSplat server: `uvicorn src.pipeline.server:app --reload --port 8000`
- ✅ Uploaded a video at `http://localhost:8000`
- ✅ Watched the progress bar complete the **Frames** and **COLMAP** stages (it will stall at Train — that is expected)
- ✅ Noted your `job_id` from `models/registry.json`

---

## Step 0 — Create your zip on the desktop

**Windows (PowerShell) — replace `<job_id>` with your actual job_id:**
```powershell
Compress-Archive -Path work\<job_id>\frames, work\<job_id>\colmap, config -DestinationPath monosplat_job.zip
```

**Mac / Linux:**
```bash
zip -r monosplat_job.zip work/<job_id>/frames work/<job_id>/colmap config/
```

The zip must contain these three things:
```
monosplat_job.zip
├── work/
│   └── <job_id>/
│       ├── frames/          ← PNG frames extracted by FFmpeg
│       └── colmap/
│           └── sparse_text/ ← cameras.txt, images.txt, points3D.txt
└── config/
    └── config.yaml
```

## Step 1 — Upload your zip to Kaggle

1. On the left sidebar click **Add Data → Upload**
2. Select your `monosplat_job.zip`
3. Wait for upload to finish — it will appear under `/kaggle/input/`
4. Run cells **top to bottom**

---

## After training — what to do on your desktop

Cell 7 will save the `.splat` and `.ply` to `/kaggle/working/`. Download them from the Output panel on the right. Then on your desktop:
1. Copy the files to `work/<job_id>/models/gaussian/`
2. Open `models/registry.json`, find your job and update:
   ```json
   "status": "ready",
   "splat_path": "work/<job_id>/models/gaussian/<job_id>.splat",
   "ply_path":   "work/<job_id>/models/gaussian/<job_id>.ply"
   ```
3. Open `http://localhost:8000/viewer/<job_id>` — your scene is live in the browser.


In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────
# You must see a GPU listed.
# If not: go to the right sidebar → Session options → Accelerator → GPU T4 x2

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅  GPU detected:')
    for line in result.stdout.split('\n')[:12]:
        print(line)
else:
    print('❌  No GPU found!')
    print('   → Right sidebar → Session options → Accelerator → GPU T4 x2')
    print('   → Then restart the session and re-run all cells.')
    raise SystemExit('No GPU. Enable GPU accelerator first.')


In [2]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────
# PyTorch is pre-installed on Kaggle. We only need a few extras.
# Takes ~30 seconds.

!pip install -q pillow pyyaml plyfile tqdm numpy

import torch
print(f'✅  PyTorch {torch.__version__}')
print(f'   CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU            : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌  CUDA not available — check your accelerator setting.')
    raise SystemExit('CUDA not available. Enable GPU T4 x2 in Session options.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
✅  PyTorch 2.10.0+cu128
   CUDA available : True
   GPU            : Tesla T4
   VRAM           : 15.6 GB


In [3]:
# ── Cell 3: Locate monosplat_job.zip ─────────────────────────────────────
# On Kaggle, files are uploaded via Add Data (left sidebar) and appear in
# /kaggle/input/. This cell finds your zip, extracts it, and sets up paths.

import zipfile, os, glob

KAGGLE_INPUT = '/kaggle/input'
WORK_DIR     = '/kaggle/working/monosplat'

# Find the uploaded zip anywhere under /kaggle/input/
zip_candidates = glob.glob(f'{KAGGLE_INPUT}/**/*.zip', recursive=True)
zip_candidates += glob.glob(f'{KAGGLE_INPUT}/*.zip')

monosplat_zip = None
for z in zip_candidates:
    if 'monosplat' in os.path.basename(z).lower() or _zip_has_work_folder(z):
        monosplat_zip = z
        break

def _zip_has_work_folder(path):
    try:
        with zipfile.ZipFile(path) as z:
            return any(n.startswith('work/') for n in z.namelist())
    except Exception:
        return False

# Re-check with helper defined above
if monosplat_zip is None:
    for z in zip_candidates:
        if _zip_has_work_folder(z):
            monosplat_zip = z
            break

if monosplat_zip is None:
    print('❌  Could not find monosplat_job.zip in /kaggle/input/')
    print()
    print('   Make sure you uploaded it via: Add Data → Upload → monosplat_job.zip')
    print(f'   Zips found: {zip_candidates if zip_candidates else "none"}')
    raise FileNotFoundError('monosplat_job.zip not found in /kaggle/input/')

zip_size = os.path.getsize(monosplat_zip) / 1e6
print(f'✅  Found zip : {monosplat_zip}  ({zip_size:.1f} MB)')

print(f'   Extracting to {WORK_DIR} ...')
os.makedirs(WORK_DIR, exist_ok=True)
with zipfile.ZipFile(monosplat_zip, 'r') as z:
    z.extractall(WORK_DIR)

os.chdir(WORK_DIR)
print(f'   Working directory: {os.getcwd()}')

# Auto-detect job_id from the work/ folder
work_dirs = sorted([d for d in glob.glob('work/*') if os.path.isdir(d)])
if not work_dirs:
    raise RuntimeError(
        '\n❌  Could not find work/<job_id>/ after extraction.'
        '\n   Make sure your zip contains: work/<job_id>/frames  work/<job_id>/colmap  config/'
    )

JOB_ID      = os.path.basename(work_dirs[0])
FRAMES_DIR  = f'work/{JOB_ID}/frames'
COLMAP_DIR  = f'work/{JOB_ID}/colmap/sparse_text'
OUTPUT_DIR  = f'work/{JOB_ID}/models/gaussian'
CKPT_DIR    = f'work/{JOB_ID}/models/checkpoints'
CONFIG_PATH = 'config/config.yaml'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

print(f'\n✅  Job ID  : {JOB_ID}')
print(f'   Frames  : {FRAMES_DIR}')
print(f'   COLMAP  : {COLMAP_DIR}')
print(f'   Output  : {OUTPUT_DIR}')
print(f'\n→  Run Cell 4 to verify COLMAP files before training.')


📂  Select your monosplat_job.zip when the picker opens...
   (This may take a moment to upload depending on file size)



Saving 4d1fe58403ff_for_colab.zip to 4d1fe58403ff_for_colab.zip

   Uploaded : 4d1fe58403ff_for_colab.zip  (577.9 MB)
   Extracting to /content/monosplat ...
   Working directory: /content/monosplat

✅  Job ID  : 4d1fe58403ff
   Frames  : work/4d1fe58403ff/frames
   COLMAP  : work/4d1fe58403ff/colmap/sparse_text
   Output  : work/4d1fe58403ff/models/gaussian

→  Run Cell 4 to verify COLMAP files before training.


In [27]:
# ── Cell 3a: (Optional) Force re-fetch src/ from GitHub ─────────────────
# Only uncomment and run this if you want to reset the src/ package to latest.
# Do NOT run this normally — it deletes your local src/ and re-clones.

# import shutil, os
# for d in ["/tmp/monosplat_src", "src", "scripts"]:
#     if os.path.exists(d):
#         shutil.rmtree(d)
#         print(f"🗑️  Removed old {d}")
# print("Re-run Cell 3b to re-fetch from GitHub.")
print("Skipped. Uncomment above only if you need to reset src/ from GitHub.")


🗑️  Removed old /tmp/monosplat_src
🗑️  Removed old src
🗑️  Removed old scripts


In [28]:
# ── Cell 3b: Bootstrap src/ package ─────────────────────────────────────
import os, subprocess, shutil, sys

src_dir     = os.path.join(os.getcwd(), "src")
scripts_dir = os.path.join(os.getcwd(), "scripts")

if os.path.isdir(src_dir):
    print("✅  src/ already present — skipping clone.")
else:
    print("📦  src/ not in zip — fetching from GitHub...")
    r1 = subprocess.run([
        "git", "clone", "--filter=blob:none", "--sparse",
        "https://github.com/Somaskandan931/monosplat.git", "/tmp/monosplat_src"
    ], capture_output=True, text=True)

    r2 = subprocess.run([
        "git", "-C", "/tmp/monosplat_src", "sparse-checkout", "set", "src", "scripts"
    ], capture_output=True, text=True)

    if r1.returncode == 0 and os.path.isdir("/tmp/monosplat_src/src"):
        shutil.copytree("/tmp/monosplat_src/src", src_dir)
        shutil.copytree("/tmp/monosplat_src/scripts", scripts_dir, dirs_exist_ok=True)
        print("✅  src/ and scripts/ fetched from GitHub.")
    else:
        print(f"git clone stderr: {r1.stderr}")
        raise RuntimeError("❌  Clone failed. Check your GitHub repo is public.")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src.reconstruction.gaussian_model import GaussianModel
print("✅  src/ importable — ready for training.")


📦  src/ not in zip — fetching from GitHub...
✅  src/ and scripts/ fetched from GitHub.
✅  src/ importable — ready for training.


In [29]:
# ── Cell 4: Verify COLMAP output ─────────────────────────────────────────
# These three files must exist and have data.
# If any are missing: COLMAP did not complete on your desktop.
# Go back to localhost:8000 and check the COLMAP stage.

import os, glob

print(f'Checking COLMAP output: {COLMAP_DIR}\n')
all_ok = True

for fname in ['cameras.txt', 'images.txt', 'points3D.txt']:
    path = os.path.join(COLMAP_DIR, fname)
    if os.path.exists(path):
        data_lines = [l for l in open(path) if not l.startswith('#') and l.strip()]
        print(f'  ✅  {fname:<18} {len(data_lines):>5} data lines')
    else:
        print(f'  ❌  {fname:<18} MISSING')
        all_ok = False

frames = (glob.glob(os.path.join(FRAMES_DIR, '*.png')) +
          glob.glob(os.path.join(FRAMES_DIR, '*.jpg')))
print(f'\n  📷  Frames : {len(frames)} files in {FRAMES_DIR}')

if not all_ok:
    print('\n❌  COLMAP did not finish. Do NOT run training yet.')
    print('   Check your desktop upload portal — COLMAP must reach 100%.')
    print('   Common causes: not enough image overlap, motion blur, poor lighting.')
elif len(frames) == 0:
    print('\n❌  No frames found. Check your zip — frames/ folder may be empty or missing.')
else:
    print('\n✅  All COLMAP files present and frames found.')
    print('   → Run Cell 5 to configure training, then Cell 6 to start.')

Checking COLMAP output: work/4d1fe58403ff/colmap/sparse_text

  ✅  cameras.txt            1 data lines
  ✅  images.txt           400 data lines
  ✅  points3D.txt        1388 data lines

  📷  Frames : 200 files in work/4d1fe58403ff/frames

✅  All COLMAP files present and frames found.
   → Run Cell 5 to configure training, then Cell 6 to start.


In [33]:
# ── Cell 5: (Optional) Adjust training iterations ─────────────────────────
# The config is already tuned for free Colab T4 (30k iters, 640x360, 200k Gaussians).
# Uncomment the FAST PREVIEW block below only if you want a quick test run first.
#
# FAST PREVIEW MODE (~15 min): uncomment to enable, then run Cell 6.
# !sed -i 's/iterations: [0-9]*/iterations: 10000/' config/config.yaml
# !sed -i 's/densify_until_iter: [0-9]*/densify_until_iter: 7000/' config/config.yaml

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
torch.cuda.empty_cache()

import yaml
with open(CONFIG_PATH) as f:
    cfg_preview = yaml.safe_load(f)

iters   = cfg_preview['training']['iterations']
est_min = iters // 750

print(f'Training configuration:')
print(f'  Iterations     : {iters:,}')
print(f'  Estimated time : ~{est_min} minutes on T4 GPU')
print(f'  Resolution     : {cfg_preview["viewer"]["window_width"]}x{cfg_preview["viewer"]["window_height"]}')
print(f'  Max Gaussians  : {cfg_preview["renderer"]["max_gaussians"]:,}')
print(f'  SH degree      : {cfg_preview["renderer"]["sh_degree"]}')
print()
print('→  When ready, run Cell 6 to start training.')


Training configuration:
  Iterations     : 30,000
  Estimated time : ~40 minutes on T4 GPU
  Max Gaussians  : 200,000
  SH degree      : 2

To change iterations: uncomment the sed line above and re-run this cell.
→  When ready, run Cell 6 to start training.


In [34]:
# (PYTORCH_ALLOC_CONF is already set in Cell 5 — this cell is kept as a safety
# re-run in case Cell 5 was skipped.)
import os, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
print("✅ Memory allocator configured.")


In [36]:
# ── Cell 6: Train Gaussian Splat model ───────────────────────────────────
import sys, os
WORK_DIR = '/kaggle/working/monosplat'
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from src.utils.config_loader import load_config
from src.preprocessing.utils import load_colmap_model
from src.reconstruction.gaussian_model import GaussianModel
from src.reconstruction.trainer import GaussianTrainer
from src.renderer.camera import Camera as ViewerCamera
from src.renderer.renderer import GaussianRenderer
import numpy as np
import torch
from PIL import Image
from pathlib import Path

# ── PATCH _knn_mean_dist with robust version ─────────────────────────────────
def _knn_mean_dist_robust(pts: torch.Tensor, k: int = 3):
    N = pts.shape[0]
    if N == 0:
        return torch.tensor([], device=pts.device)
    k_actual = min(k, N - 1)
    if k_actual == 0:
        return torch.zeros(N, device=pts.device)
    dist_matrix = torch.cdist(pts, pts)
    dist_matrix.fill_diagonal_(float('inf'))
    knn_dists = torch.topk(dist_matrix, k=k_actual, largest=False, dim=1).values
    return knn_dists.mean(dim=1).clamp(min=1e-7)

try:
    from src.reconstruction import gaussian_model
    if hasattr(gaussian_model, '_knn_mean_dist'):
        gaussian_model._knn_mean_dist = _knn_mean_dist_robust
        print('✅ Patched _knn_mean_dist')
except Exception as e:
    print(f'⚠️  Could not patch _knn_mean_dist: {e}')

# ── LOAD CONFIG ───────────────────────────────────────────────────────────────
cfg = load_config("config/config.yaml")

# Memory-safe caps — respect config but never exceed free-tier T4 limits
cfg.training.iterations    = min(cfg.training.iterations,    30000)
cfg.renderer.max_gaussians = min(cfg.renderer.max_gaussians, 200000)
cfg.renderer.sh_degree     = min(cfg.renderer.sh_degree,     1)

print(f"Config: iterations={cfg.training.iterations:,}, "
      f"max_gaussians={cfg.renderer.max_gaussians:,}, "
      f"sh_degree={cfg.renderer.sh_degree}, "
      f"resolution={cfg.viewer.window_width}x{cfg.viewer.window_height}")

# ── LOAD COLMAP DATA ──────────────────────────────────────────────────────────
cameras_colmap, images_colmap, points3d = load_colmap_model(COLMAP_DIR)

# ── PREPARE TRAINING DATA (images stay on CPU to save VRAM) ──────────────────
W, H = cfg.viewer.window_width, cfg.viewer.window_height
train_cameras, train_images = [], []

for img_data in images_colmap.values():
    cam_data = cameras_colmap[img_data.camera_id]
    cam      = ViewerCamera.from_colmap(img_data, cam_data, W, H)
    img_path = Path(FRAMES_DIR) / img_data.name
    if not img_path.exists():
        img_path = Path(FRAMES_DIR) / Path(img_data.name).name
    if img_path.exists():
        img = np.array(
            Image.open(img_path).convert("RGB").resize((W, H), Image.LANCZOS),
            dtype=np.float32
        ) / 255.0
        train_cameras.append(cam)
        train_images.append(torch.from_numpy(img).permute(2, 0, 1).cpu())
    else:
        print(f"⚠️  Skipping missing image: {img_data.name}")

print(f"Loaded {len(train_cameras)} cameras, {len(points3d):,} 3D points")
if len(train_cameras) == 0:
    raise RuntimeError("No training images loaded. Check FRAMES_DIR and COLMAP_DIR.")

# ── PATCH GaussianModel ops to run under no_grad ─────────────────────────────
def _make_nograd(fn):
    def _wrapped(*a, **kw):
        with torch.no_grad():
            return fn(*a, **kw)
    return _wrapped

for method in ["reset_opacity", "densify_and_prune", "prune_points"]:
    if hasattr(GaussianModel, method):
        orig = getattr(GaussianModel, method)
        setattr(GaussianModel, method, _make_nograd(orig))
        print(f"✅ Patched GaussianModel.{method}")

# ── INITIALISE MODEL ──────────────────────────────────────────────────────────
xyzs = np.array([p.xyz for p in points3d.values()], dtype=np.float32)
rgbs = np.array([p.rgb for p in points3d.values()], dtype=np.float32) / 255.0

model = GaussianModel(sh_degree=cfg.renderer.sh_degree)
model.create_from_points(xyzs, rgbs)

# ── RENDERER + TRAINER ────────────────────────────────────────────────────────
torch.cuda.empty_cache()

renderer = GaussianRenderer(width=W, height=H, device="cuda", batch_size=5000)
trainer  = GaussianTrainer(model, renderer.render_torch, train_cameras, train_images, cfg)

# ── TRAIN ─────────────────────────────────────────────────────────────────────
print(f"\nStarting training for {cfg.training.iterations:,} iterations...")
trainer.train()

print("\n✅ Training complete!")


✅ Patched src.reconstruction.gaussian_model._knn_mean_dist with robust version.
Config loaded: iterations=30000, max_gaussians=200000
Effective: iterations=30000, max_gaussians=200000, sh_degree=1
✅ Auto-detected scene: 4d1fe58403ff
   colmap_dir : work/4d1fe58403ff/colmap/sparse_text
   image_dir  : work/4d1fe58403ff/frames
[colmap] Loaded model: 1 cameras, 200 images, 1,388 3D points
Loaded 200 cameras
✅ Patched GaussianModel.prune_points
Point cloud: 1,388 points
[GaussianModel] Initialised 1,388 Gaussians from point cloud.
[Trainer] Device : cuda
[Trainer] Output : /content/monosplat/models/gaussian
[Trainer] Ckpts  : /content/monosplat/models/checkpoints

Starting training for 30000 iterations...


Training:   0%|          | 0/30000 [00:05<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 12.70 GiB is allocated by PyTorch, and 1.73 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ── NOTE: This cell is intentionally disabled ────────────────────────────
# Training is handled entirely by Cell 6 above (the Python cell).
# The `!python -m scripts.train` call below is kept only as a reference
# for running training locally from a terminal — do not run it here.
#
# !python -m scripts.train \
#     --config     "{CONFIG_PATH}" \
#     --colmap_dir "{COLMAP_DIR}" \
#     --image_dir  "{FRAMES_DIR}" \
#     --output_dir "{OUTPUT_DIR}"

print("Training is handled by Cell 6. This cell is a no-op.")


In [ ]:
# ── Cell 7: Locate output files ──────────────────────────────────────────
# On Kaggle, files saved to /kaggle/working/ are available in the
# Output panel on the right side of the screen.
# This cell copies the final .splat and .ply to /kaggle/working/ for download.

import glob, os, shutil

outputs = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, '*.ply')) +
    glob.glob(os.path.join(OUTPUT_DIR, '*.splat')),
    key=os.path.getmtime
)

if not outputs:
    print('❌  No output files found. Make sure Cell 6 completed successfully.')
else:
    downloaded = []
    for ext in ['.splat', '.ply']:
        matching = [f for f in outputs if f.endswith(ext)]
        if matching:
            latest   = matching[-1]
            dst_name = f'{JOB_ID}{ext}'
            dst_path = f'/kaggle/working/{dst_name}'
            shutil.copy2(latest, dst_path)
            size_mb  = os.path.getsize(dst_path) / 1e6
            print(f'  ✅  {dst_name}  ({size_mb:.1f} MB)  →  /kaggle/working/')
            downloaded.append(dst_name)

    print(f'\n✅  Files ready: {", ".join(downloaded)}')
    print()
    print('─' * 60)
    print('HOW TO DOWNLOAD from Kaggle:')
    print('─' * 60)
    print('  1. Look at the right sidebar → Output panel')
    print('  2. You will see the .splat and .ply files listed')
    print('  3. Click the download icon next to each file')
    print()
    print('─' * 60)
    print('NEXT STEPS on your desktop:')
    print('─' * 60)
    print(f'  1. Create this folder if it does not exist:')
    print(f'       work\\{JOB_ID}\\models\\gaussian\\')
    print()
    print(f'  2. Move the downloaded files there:')
    print(f'       {JOB_ID}.splat  →  work\\{JOB_ID}\\models\\gaussian\\')
    print(f'       {JOB_ID}.ply    →  work\\{JOB_ID}\\models\\gaussian\\')
    print()
    print(f'  3. Open models/registry.json and find the entry for job {JOB_ID}.')
    print(f'     Update these three fields:')
    print(f'       "status":     "ready",')
    print(f'       "splat_path": "work/{JOB_ID}/models/gaussian/{JOB_ID}.splat",')
    print(f'       "ply_path":   "work/{JOB_ID}/models/gaussian/{JOB_ID}.ply"')
    print()
    print(f'  4. Make sure your local server is running:')
    print(f'       uvicorn src.pipeline.server:app --reload --port 8000')
    print()
    print(f'  5. Open in your browser:')
    print(f'       http://localhost:8000/viewer/{JOB_ID}')
    print()
    print('  Or drag the .splat directly to: https://supersplat.playcanvas.com')
